In [7]:
!pip install pypdf langchain-text-splitters

In [8]:
import os
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\SIC\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
DATA_PATH = "../data"

documents = []

for disease in os.listdir(DATA_PATH):

    disease_path = os.path.join(DATA_PATH, disease)

    if not os.path.isdir(disease_path):
        continue

    for filename in os.listdir(disease_path):

        if filename.lower().endswith(".pdf"):

            file_path = os.path.join(disease_path, filename)

            reader = PdfReader(file_path)

            text = ""

            for page in reader.pages:
                page_text = page.extract_text()

                if page_text:
                    text += page_text + "\n"

            documents.append({
                "disease": disease,
                "filename": filename,
                "text": text
            })

print("Total articles:", len(documents))

Total articles: 14


In [10]:
print("Disease:", documents[0]["disease"])
print("File:", documents[0]["filename"])
print(documents[0]["text"][:2000])

Disease: atelectasis
File: Atelectasis.pdf
R E S P I R A T I O N A N D T H E A I R W A Y
Hypotension and pulmonary atelectasis during general anaesthesia: 
a randomised crossover laboratory study
Erland €Ostberg 1,2, *
 , Cecilia Wassberg 3 , Mark Lubberink 3,4 , Jonathan Sigfridsson 3
 ,
Mikl� os Lipcsey 5,6
 , Gaetano Perchiazzi 6,7 , Philippe Wagner 2 and Lennart Edmark 2
1 Department of Anaesthesia and Intensive Care, V€ astmanland Hospital V€ astera˚ s, Sweden, 2 Centre for Clinical Research 
V€ astmanland, Department of Surgical Sciences, Uppsala University, Sweden, 3 Molecular Imaging and Medical Physics, 
Department of Surgical Sciences, Uppsala University, Sweden, 4 Uppsala Centre for Molecular Imaging and Theranostics, 
Department of Surgical Sciences, Uppsala University, Sweden, 5 Anaesthesiology and Intensive Care, Department of 
Surgical Sciences, Uppsala University, Sweden, 6 Hedenstierna Laboratory, Department of Surgical Sciences, Uppsala 
University, Sweden and 7 Depar

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for doc in documents:

    doc_chunks = splitter.split_text(doc["text"])

    for chunk in doc_chunks:

        chunks.append({
            "disease": doc["disease"],
            "filename": doc["filename"],
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total chunks: 730


In [12]:
print(chunks[0])

{'disease': 'atelectasis', 'filename': 'Atelectasis.pdf', 'text': 'R E S P I R A T I O N A N D T H E A I R W A Y\nHypotension and pulmonary atelectasis during general anaesthesia: \na randomised crossover laboratory study\nErland €Ostberg 1,2, *\n , Cecilia Wassberg 3 , Mark Lubberink 3,4 , Jonathan Sigfridsson 3\n ,\nMikl� os Lipcsey 5,6\n , Gaetano Perchiazzi 6,7 , Philippe Wagner 2 and Lennart Edmark 2\n1 Department of Anaesthesia and Intensive Care, V€ astmanland Hospital V€ astera˚ s, Sweden, 2 Centre for Clinical Research \nV€ astmanland, Department of Surgical Sciences, Uppsala University, Sweden, 3 Molecular Imaging and Medical Physics, \nDepartment of Surgical Sciences, Uppsala University, Sweden, 4 Uppsala Centre for Molecular Imaging and Theranostics, \nDepartment of Surgical Sciences, Uppsala University, Sweden, 5 Anaesthesiology and Intensive Care, Department of \nSurgical Sciences, Uppsala University, Sweden, 6 Hedenstierna Laboratory, Department of Surgical Sciences, Upp

In [13]:
!pip install sentence-transformers

In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6388.65it/s]


In [15]:
texts = [chunk["text"] for chunk in chunks]

In [16]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding size:", embeddings.shape[1])

Batches: 100%|██████████| 23/23 [00:07<00:00,  3.10it/s]

Number of embeddings: 730
Embedding size: 384


In [17]:
!pip install faiss-cpu

In [18]:
import faiss
import numpy as np

In [19]:
embedding_matrix = np.array(embeddings).astype("float32")

In [20]:
index = faiss.IndexFlatL2(embedding_matrix.shape[1])

index.add(embedding_matrix)

print("Total vectors in database:", index.ntotal)

Total vectors in database: 730


In [21]:
import faiss

faiss.write_index(index, "../vector_db/medical_index.faiss")

In [22]:
import pickle

with open("../vector_db/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

In [23]:
import faiss
import pickle
import numpy as np

index = faiss.read_index("../vector_db/medical_index.faiss")

with open("../vector_db/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

print("Vectors:", index.ntotal)
print("Chunks:", len(chunks))

Vectors: 730
Chunks: 730


In [24]:
def retrieve(query, k=5):

    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []

    for i in indices[0]:
        results.append(chunks[i])

    return results

In [25]:
query = "What are the common symptoms and clinical features of pneumonia?"

results = retrieve(query, k=5)

for i, result in enumerate(results):
    print("\n--- Result", i + 1, "---")
    print("Disease:", result["disease"])
    print("Article:", result["filename"])
    print(result["text"][:500])


--- Result 1 ---
Disease: pneumonia
Article: pneumonia2.pdf
M. pneumoniae etiologies [1,2,4–10,37,38] (Fig. 1).
CAP CO-INFECTIONS
Because co-infections, i.e., dual typical bacterial
pathogens, dual atypical pathogens, or typical
plus an atypical pathogen, are exceedingly rare
there is no need to expend diagnostic resources
looking for copathogens in patients with CAP.
Studies that have reported typical or atypical
copathogens have been based on culture for one
and serology for the other. Serological diagnoses,
particularly those based upon IgG titres, ar

--- Result 2 ---
Disease: pneumonia
Article: pneumonia1.pdf
pneumonia and distinguish bacterial from viral etiology, and to determine clinical and labora-
tory predictors of bacterial pneumonia.
Methods
Children 3 months to 16 years of age presenting to seven Canadian pediatric EDs before
the COVID-19 pandemic with fever and cough who had a chest radiograph performed for
possible pneumonia were enrolled and underwent standardized cli

In [26]:
questions = [
    "What are the common clinical features of pneumonia?",
    "What are the causes of pneumonia?",
    "How is pneumonia diagnosed?"
]

for question in questions:
    print("\nQUESTION:", question)

    results = retrieve(question, k=3)

    for i, result in enumerate(results):
        print(f"\nResult {i+1}:")
        print("Disease:", result["disease"])
        print("Article:", result["filename"])
        print(result["text"][:300])


QUESTION: What are the common clinical features of pneumonia?

Result 1:
Disease: pneumonia
Article: pneumonia2.pdf
M. pneumoniae etiologies [1,2,4–10,37,38] (Fig. 1).
CAP CO-INFECTIONS
Because co-infections, i.e., dual typical bacterial
pathogens, dual atypical pathogens, or typical
plus an atypical pathogen, are exceedingly rare
there is no need to expend diagnostic resources
looking for copathogens in patients

Result 2:
Disease: pneumonia
Article: pneumonia2.pdf
conﬁrm or rule out speciﬁc pathogens [1–4,7–10].
In the literature, most studies have been unable
to clearly differentiate typical from atypical pneu-
monias. The main difﬁculty with such studies is
that they have compared individual clinical and
laboratory ﬁndings of atypical and typical path-
ogen

Result 3:
Disease: pulmonary infiltrate
Article: pulmonary infiltrate2.pdf
O. Klinkova, MD (*) 
Moffitt Cancer Center, Tampa, FL, USA
e-mail: olga.klinkova@moffitt.org
24
and intensity of immunosuppression should be considered

In [27]:
def create_prompt(question, results):

    context = "\n\n".join(
        [result["text"] for result in results]
    )

    prompt = f"""
You are a medical clinical decision support assistant.

Answer the question using ONLY the provided context.
If the answer is not present in the context, say:
"Information not found in the provided medical sources."

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [28]:
question = "What are the common clinical features of pneumonia?"

results = retrieve(question, k=5)

prompt = create_prompt(question, results)

print(prompt)


You are a medical clinical decision support assistant.

Answer the question using ONLY the provided context.
If the answer is not present in the context, say:
"Information not found in the provided medical sources."

Context:
M. pneumoniae etiologies [1,2,4–10,37,38] (Fig. 1).
CAP CO-INFECTIONS
Because co-infections, i.e., dual typical bacterial
pathogens, dual atypical pathogens, or typical
plus an atypical pathogen, are exceedingly rare
there is no need to expend diagnostic resources
looking for copathogens in patients with CAP.
Studies that have reported typical or atypical
copathogens have been based on culture for one
and serology for the other. Serological diagnoses,
particularly those based upon IgG titres, are
Table 1. Diagnostic features of the
non-zoonotic atypical pneumonias.
Adapted from Cunha, 2006 [6] Key Characteristics
Mycoplasma
pneumoniae
Legionnaire’s
disease
Chlamydophilia
(Chlamydia)
pneumoniae
Symptoms
Mental confusion ±a +–
Prominent headache – ± –
Meningismus –

In [29]:
!pip install ollama

In [35]:
import ollama

def generate_answer(question, results):

    context = "\n\n".join([result["text"] for result in results])

    prompt = f"""
You are a medical information assistant.

Answer the patient's question using ONLY the provided medical context.

Your answer must:
- Use very simple language that a patient or family member can understand.
- Avoid complex medical terminology. If a medical term is necessary, explain it simply.
- Explain what the disease is, what the patient can do, and what treatment may be used.
- Give practical and clear guidance.
- Use short bullet points.
- Do NOT overwhelm the patient with drug names, dosages, or highly technical details unless specifically asked.
- Do NOT invent information that is not in the provided context.
- Do NOT make a diagnosis.
- Do NOT claim that the information guarantees a cure.
- For treatment decisions, advise the patient to consult a qualified doctor.

Medical context:
{context}

Patient's question:
{question}

Give a simple, patient-friendly answer:
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response["message"]["content"]

In [33]:
question = "What are the common clinical features of pneumonia?"

results = retrieve(question, k=5)

answer = generate_answer(question, results)

print(answer)

The common clinical features of pneumonia are summarized in Table 1, adapted from Cunha, 2006 [6]. The diagnostic features of non-zoonotic atypical pneumonias are as follows:

- Mental confusion
- Prominent headache
- Meningismus
- Myalgias
- Ear pain
- Pleuritic pain
- Abdominal pain
- Diarrhea
- Rash (non-exudative pharyngitis)
- Non-exudative pharyngitis
- Haemoptysis
- Lobar consolidation
- Cardiac involvement

It is worth noting that there are few discernible differences in isolated findings between typical and atypical pathogens, and that clinical and laboratory findings alone are not sufficient to clearly differentiate between the two. However, using a weighted syndromic approach based on the relative clinical specificity of characteristic clinical findings, it is possible to differentiate typical from atypical pneumonias with good sensitivity and specificity.


In [34]:
question = "how to treat pneumonia?"

results = retrieve(question, k=5)

answer = generate_answer(question, results)

print(answer)

Based on the provided context, the treatment of pneumonia depends on the etiologic agent. For typical bacterial pneumonia, antibiotics appear to hasten resolution, prevent almost all deaths, and may decrease the incidence of complications. Antibiotics play no role in the management of viral pneumonia.

The recommended treatment options are as follows:

- For viral pneumonia, the treatment is based on the specific etiologic agent:
  - CMV pneumonia: Intravenous gancyclovir or foscarnet with a combination of intravenous immunoglobulin or CMV-specific immunoglobulin.
  - Influenza A and B infections: Neuraminidase inhibitors, such as oseltamivir, zanamivir, or peramivir.
  - RSV pneumonia: Aerosolized, oral, or intravenous ribavirin, with the addition of intravenous immunoglobulin to reduce mortality.

- For bacterial pneumonia, antibiotics are effective in hastening resolution and preventing deaths:
  - Typical bacterial pneumonia: Antibiotics, although the specific antibiotics are not l

In [36]:
question = "How can pneumonia be treated and what care does the patient need?"

results = retrieve(question, k=5)

answer = generate_answer(question, results)

print(answer)

Here's a simple and patient-friendly answer:

Pneumonia is a type of infection that inflames the air sacs in the lungs. It can be caused by bacteria, viruses, or other germs. The good news is that pneumonia is usually treatable with medicine.

Here are some general things you can do to help your body fight off the infection:

• Stay in bed and rest as much as you can
• Drink plenty of fluids, like water, tea, or broth
• Use a humidifier to add moisture to the air, which can help loosen up mucus
• Take medication as prescribed by your doctor to help clear up the infection
• If you have a fever, you can take medicine to help lower it

However, if you have a severe case of pneumonia or if you're not getting better, you may need to go to the hospital for more treatment. In some cases, you may need oxygen therapy to help your body get enough oxygen.

It's also important to note that antibiotics won't work against viral pneumonia. Your doctor will be able to tell you what type of treatment y